In [1]:
"""
Build basis_master sheet: fc_family_composition
-----
FC parents/in-laws by age band (step function).
Source: Basis and Assumptions.xlsx > FC Basis > BG3:BJ13.

Note: col BH ('Premium per Member') is LEGACY — intentionally dropped.

FC row-4 formula wiring (V1.7.43):
  AE (# Parents) = VLOOKUP(G4, FC_Parents, 3, TRUE)   -> BI column
  AF (# In-Laws) = VLOOKUP(G4, FC_Parents, 4, TRUE)   -> BJ column
Named range FC_Parents = 'FC Basis'!$BG$4:$BJ$13.

Author: Mohamed Hussein Gadiwala
Basis version: v2026_07 (basis master)
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import csv
import openpyxl
import polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")

SOURCE_XLSX  = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "FC Basis"

BASIS_VERSION = "v2026_07"
SNAPSHOT_DIR = BASE / "basis_snapshots" / f"basis_{BASIS_VERSION}"
MASTER_XLSX  = BASE / "basis_master" / f"basis_master_{BASIS_VERSION}.xlsx"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME   = "fc_family_composition"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")
print(f"Target:   {PARQUET_PATH}")
print(f"Built at: {BUILT_AT}")


Building: fc_family_composition
Source:   Basis and Assumptions.xlsx > FC Basis
Target:   J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07\fc_family_composition.parquet
Built at: 2026-07-22T10:49:35+00:00


In [2]:
def find_all_cells_any_col(ws, label: str, max_row: int = 1200, max_col: int = 30) -> list[tuple[int, int]]:
    """Find ALL cells matching `label` exactly, anywhere on the sheet."""
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits


def find_in_range(ws, label: str, row_start: int, row_end: int, col: int | None = None) -> tuple[int, int] | None:
    """Find first cell matching `label` within given row range (and optional col)."""
    hits = find_all_cells_any_col(ws, label)
    for r, c in hits:
        if row_start <= r < row_end and (col is None or c == col):
            return (r, c)
    return None


def get_col_letter(col: int) -> str:
    return openpyxl.utils.get_column_letter(col)


In [3]:
# BG=59, BH=60 (legacy, dropped), BI=61, BJ=62
COL_AGE, COL_PARENTS, COL_INLAWS = 59, 61, 62

header_age     = ws.cell(row=3, column=COL_AGE).value
header_parents = ws.cell(row=3, column=COL_PARENTS).value
header_inlaws  = ws.cell(row=3, column=COL_INLAWS).value

assert isinstance(header_age, str) and "age" in header_age.lower()
assert isinstance(header_parents, str) and "parent" in header_parents.lower()
assert isinstance(header_inlaws, str) and "law" in header_inlaws.lower()

DATA_ROW_START = 4
DATA_ROW_END   = 14
EXPECTED_AGES  = [0, 25, 30, 35, 40, 45, 50, 55, 60, 65]

all_rows: list[dict] = []
for excel_row in range(DATA_ROW_START, DATA_ROW_END):
    age       = ws.cell(row=excel_row, column=COL_AGE).value
    n_parents = ws.cell(row=excel_row, column=COL_PARENTS).value
    n_in_laws = ws.cell(row=excel_row, column=COL_INLAWS).value

    assert isinstance(age, (int, float))
    assert isinstance(n_parents, (int, float))
    assert isinstance(n_in_laws, (int, float))

    all_rows.append({
        "product":   "fc",
        "age_from":  int(age),
        "n_parents": float(n_parents),
        "n_in_laws": float(n_in_laws),
    })

print(f"Total rows harvested: {len(all_rows)}")
print("(source col BH 'Premium per Member' dropped — legacy)")
for r in all_rows:
    print(f"  age_from={r['age_from']:>2d}  n_parents={r['n_parents']}  n_in_laws={r['n_in_laws']}")


Total rows harvested: 10
(source col BH 'Premium per Member' dropped — legacy)
  age_from= 0  n_parents=2.0  n_in_laws=0.0
  age_from=25  n_parents=2.0  n_in_laws=2.0
  age_from=30  n_parents=2.0  n_in_laws=2.0
  age_from=35  n_parents=2.0  n_in_laws=2.0
  age_from=40  n_parents=1.5  n_in_laws=1.5
  age_from=45  n_parents=1.5  n_in_laws=1.5
  age_from=50  n_parents=1.0  n_in_laws=1.0
  age_from=55  n_parents=1.0  n_in_laws=1.0
  age_from=60  n_parents=0.5  n_in_laws=0.5
  age_from=65  n_parents=0.5  n_in_laws=0.5


In [4]:
df = pl.DataFrame(all_rows).select([
    "product", "age_from", "n_parents", "n_in_laws",
])

dupes = df.group_by(["product", "age_from"]).len().filter(pl.col("len") > 1)
assert dupes.height == 0

assert set(df["product"].unique().to_list()) == {"fc"}

ages = sorted(df["age_from"].to_list())
assert ages == sorted(EXPECTED_AGES)

assert df["n_parents"].min() >= 0 and df["n_parents"].max() <= 5
assert df["n_in_laws"].min() >= 0 and df["n_in_laws"].max() <= 5

def get_row(age_from):
    return df.filter(pl.col("age_from") == age_from).row(0, named=True)

r = get_row(0);  assert r["n_parents"] == 2.0 and r["n_in_laws"] == 0.0
r = get_row(25); assert r["n_parents"] == 2.0 and r["n_in_laws"] == 2.0
r = get_row(40); assert r["n_parents"] == 1.5 and r["n_in_laws"] == 1.5
r = get_row(65); assert r["n_parents"] == 0.5 and r["n_in_laws"] == 0.5

parents_by_age = df.sort("age_from")["n_parents"].to_list()
for i in range(1, len(parents_by_age)):
    assert parents_by_age[i] <= parents_by_age[i-1]

print(f"✓ {df.height} rows validated")
print(f"✓ age bands: {ages}")
print(f"✓ 8 frozen numeric checks passed")
print(f"✓ n_parents monotonic non-increasing")
print(df)


✓ 10 rows validated
✓ age bands: [0, 25, 30, 35, 40, 45, 50, 55, 60, 65]
✓ 8 frozen numeric checks passed
✓ n_parents monotonic non-increasing
shape: (10, 4)
┌─────────┬──────────┬───────────┬───────────┐
│ product ┆ age_from ┆ n_parents ┆ n_in_laws │
│ ---     ┆ ---      ┆ ---       ┆ ---       │
│ str     ┆ i64      ┆ f64       ┆ f64       │
╞═════════╪══════════╪═══════════╪═══════════╡
│ fc      ┆ 0        ┆ 2.0       ┆ 0.0       │
│ fc      ┆ 25       ┆ 2.0       ┆ 2.0       │
│ fc      ┆ 30       ┆ 2.0       ┆ 2.0       │
│ fc      ┆ 35       ┆ 2.0       ┆ 2.0       │
│ fc      ┆ 40       ┆ 1.5       ┆ 1.5       │
│ fc      ┆ 45       ┆ 1.5       ┆ 1.5       │
│ fc      ┆ 50       ┆ 1.0       ┆ 1.0       │
│ fc      ┆ 55       ┆ 1.0       ┆ 1.0       │
│ fc      ┆ 60       ┆ 0.5       ┆ 0.5       │
│ fc      ┆ 65       ┆ 0.5       ┆ 0.5       │
└─────────┴──────────┴───────────┴───────────┘


In [5]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name} ({PARQUET_SIZE_KB:.2f} KB)")

if MASTER_XLSX.exists():
    master_wb = openpyxl.load_workbook(MASTER_XLSX)
else:
    master_wb = openpyxl.Workbook()
    if "Sheet" in master_wb.sheetnames:
        del master_wb["Sheet"]

if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated")


✓ Wrote fc_family_composition.parquet (1.51 KB)
✓ Master workbook updated


In [6]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "basis_version", "notes",
]

log_row = {
    "built_at": BUILT_AT, "built_by": BUILT_BY, "sheet": SHEET_NAME,
    "rows": df.height, "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}", "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name, "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA, "basis_version": BASIS_VERSION,
    "notes": ("FC parents/in-laws by age (10 bands, 0..65 step-5). "
              "Named range FC_Parents. Col BH 'Premium per Member' dropped (legacy). "
              "Consumer: FC row-4 AE/AF."),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}\nBUILD COMPLETE: {SHEET_NAME}\n{'='*60}")


✓ build_log.csv updated

BUILD COMPLETE: fc_family_composition
